In [1]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)

from fixedincomelib import *

# FRE-GY 9743 - Assignment 1

**Name:** Lei Zhao  
**NetID:** lz2665

## Part I - One-Dimensional Interpolation

This section implements and validates a piecewise-constant left-continuous interpolator with flat extrapolation. It covers point interpolation, exact integration, sensitivity to each ordinate, and sensitivity of the integrated value. Analytic gradients are compared with bump-and-revalue calculations.


### Test interpolation

In [2]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3, expected 3.0, diff = 0.0
f(1.0) = 3, expected 3.0, diff = 0.0
f(1.5) = 4, expected 4.0, diff = 0.0
f(3.0) = 4, expected 4.0, diff = 0.0
f(5.5) = 6, expected 6.0, diff = 0.0
f(6.5) = 6, expected 6.0, diff = 0.0
f(8.0) = 6, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [3]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [4]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    if bump_size == 0:
        raise ValueError("bump_size must be non-zero")

    base_interpolator = qfCreate1DInterpolator(
        axis1, values, interp_method, extrap_method)
    base_value = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = []
    for i in range(len(values)):
        bumped_values = np.array(values, dtype=float, copy=True)
        bumped_values[i] += bump_size
        bumped_interpolator = qfCreate1DInterpolator(
            axis1, bumped_values, interp_method, extrap_method)
        bumped_value = qfInterpolate1DIntegral(
            x_s, x_e, bumped_interpolator)
        grad.append((bumped_value - base_value) / bump_size)

    return np.array(grad)

### Interpolation sensitivity

In [5]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [6]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 1.659827830735594e-11
[1.5, 5.2]: max abs diff = 2.1259438653942198e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 7.571543392259628e-11


# Bond Forward Business - Written Answers

Notation used below: $N=\$1{,}000{,}000$ is the Treasury face amount, $T_s$ is the forward settlement date, $T_m$ is the bond maturity, $P(t)$ is the bond's time-$t$ full (dirty) spot price, and $K$ is the delivery price agreed at inception. Quoted clean prices must be converted to full prices by adding accrued interest before applying the pricing equations.

## Q1. Client motivation, term sheet, and settlement

A client that expects to buy the Treasury in one year may use the forward to lock in its future purchase price and yield today. This removes the risk that Treasury prices rise (or yields fall) before the planned purchase. The trade can also be used to add future duration exposure without paying for the bond today, match a future liability or cash inflow, express a directional view on rates, or avoid the operational and balance-sheet cost of holding and financing the cash bond for the next year.

Important term-sheet fields include: trade date; buyer and seller; $N$; exact deliverable Treasury (CUSIP, coupon, maturity and any substitution rights); $T_s$; fixed forward clean or dirty price $K$ and its quotation per $100$ face; treatment of accrued interest and coupons; business-day convention and settlement calendar; physical versus cash settlement; payment and delivery instructions; CSA/collateral currency and discounting terms; events of default, termination and settlement-failure provisions.

For physical settlement, Bank A is short the forward: on $T_s$ it delivers $1{,}000{,}000$ face amount of the specified Treasury. The client pays the contractual delivery amount. If $K$ is a clean price quoted per $100$ face, the cash amount is $N(K+AI(T_s))/100$; if $K$ is already a full price, it is $NK/100$. Coupons paid before $T_s$ do not belong to the forward buyer. For cash settlement, the client receives $N[B(T_s;T_s,T_m)-K]/100$ (and pays Bank A if this signed amount is negative), subject to the contract's price convention.

## Q2. Cash-bond hedge and internal funding

The client is long the forward, so Bank A is short: a rise in the bond's settlement-date price increases Bank A's forward liability. Buying the exact deliverable bond at inception gives the desk the asset it must deliver and offsets the forward's major bond-price and duration exposure. It also turns the problem into a known cash-and-carry calculation rather than a forecast of the future bond price.

The rates desk buys the cash bond for $P(0)$. It then delivers the bond to the repo desk as collateral and borrows the purchase cash through a term repo maturing on $T_s$. Economically, the repo desk supplies secured financing and receives the collateral; the treasury desk supplies or charges the bank's underlying cash/funding to the repo desk through the internal transfer-pricing system. Thus the internal chain is Treasury desk $\rightarrow$ Repo desk $\rightarrow$ Rates desk for cash, with the bond collateral flowing from the Rates desk to the Repo desk. Coupon payments on collateral during the repo are credited to the cash-bond owner through the repo's coupon/manufactured-payment mechanics and therefore reduce the net financing requirement.

## Q3. Forward price from spot price and repo

Let $C_i$ be each coupon paid at $t_i\leq T_s$, and let $DF_R(a,b)$ denote the discount factor implied by the applicable repo financing curve. The no-arbitrage full forward price is

$$B(0;T_s,T_m)=\frac{P(0)-\sum_{0<t_i\leq T_s}C_iDF_R(0,t_i)}{DF_R(0,T_s)}.$$

This says that the financed cost carried to settlement equals the initial full bond price minus the present value of coupons received before settlement. With a constant simple repo rate $R$ and year fractions $\tau(a,b)$, the same replication can be written

$$B(0;T_s,T_m)=P(0)[1+R\tau(0,T_s)]-\sum_{t_i\leq T_s}C_i[1+R\tau(t_i,T_s)].$$

If rates are continuously compounded, replace each accumulation factor by $e^{R(T_s-t)}$. The fair inception strike is $K=B(0;T_s,T_m)$ before adding any dealer spread. Exact implementation must use the bond's coupon dates, day-count convention and dirty-price treatment.

## Q4. Mark-to-market during the life of the trade

At any $t\in(0,T_s]$, first recompute the prevailing full forward bond price using the current spot bond price, the coupons remaining between $t$ and $T_s$, and the current repo curve:

$$B(t;T_s,T_m)=\frac{P(t)-\sum_{t<t_i\leq T_s}C_iDF_R(t,t_i)}{DF_R(t,T_s)}.$$

The value to the client (the long) is then

$$V_{client}(t)=\frac{N}{100}DF_{CSA}(t,T_s)\left[B(t;T_s,T_m)-K\right],$$

with consistent clean/dirty units. Bank A's value is the negative of this amount. On $T_s$, the discount factor is one and the value becomes the signed difference between the market full price and the delivery price.

## Q5. Residual risk and dealer revenue

Buying the exact bond and locking term repo to $T_s$ removes the principal outright Treasury-price risk: the desk already owns the asset required for delivery and has largely fixed its carry. It does not remove every risk. Residual exposures include repo/funding basis if the financing is not perfectly locked, repo haircut and margin liquidity, counterparty and settlement-fail risk, CSA-versus-repo discounting basis, coupon-reinvestment details, balance-sheet costs and operational risk. An imperfect deliverable or maturity mismatch would also introduce bond basis risk.

Under ideal no-arbitrage assumptions and identical financing terms, the replicated fair price produces no economic profit. The desk earns revenue by charging a bid-offer or embedding a financing spread in the client forward price. For a client buying the bond forward, Bank A can quote a delivery price based on a client repo rate above the bank's actual internal secured funding rate, subject to competition and fair-value controls. The client may still accept because the forward provides future duration exposure without immediate cash payment, repo access, collateral management, custody or balance-sheet usage. The spread compensates the bank for intermediation, capital, liquidity and residual risks rather than for an unhedged directional bet.

## *Q5. Connection to risk-neutral pricing

The cash-and-carry replication and risk-neutral valuation are two expressions of the same no-arbitrage condition. Under frictionless trading, no default or settlement failure, consistent collateral/repo treatment, and the ability to finance the bond and reinvest coupons at the modeled rates, replication gives

$$P(t)=\sum_{t<t_i\leq T_s}C_iDF_R(t,t_i)+DF_R(t,T_s)B(t;T_s,T_m).$$

Solving this identity gives exactly the Q3 forward-price formula. In a risk-neutral formulation, the value of the settlement payoff is its conditional expectation under the pricing measure discounted using the collateral-consistent numeraire. Equivalently, under the $T_s$-settlement numeraire, the forward bond price is the appropriate conditional expectation of the bond's $T_s$ value. When repo is the effective carry curve for the bond and the above assumptions hold, that expectation must equal the replicating cash-and-carry price; otherwise an arbitrage would exist. CSA discounting discounts the derivative payoff, while the repo curve determines the bond's financed carry. Funding, collateral or repo basis effects must therefore be modeled consistently when those ideal assumptions do not hold.